# Notebook 02 — RQ2 & RQ3: Partition Hierarchy Depth and Key Ordering

Covers:
- **RQ2**: How hierarchy depth (B1–B4) affects selective query performance
- **RQ3**: Whether partition key ordering (B3 vs C1) impacts latency

Produces:
- Table 2 (paper): hierarchy depth results, SF-10
- Table 3 (paper): key-ordering results, SF-10
- Figure: B1–B4 speedup / slowdown bars (SF-10 and SF-1 side-by-side)
- Figure: C1 key-ordering comparison with B3 reference
- Wilcoxon tests for all hierarchy and ordering comparisons


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import wilcoxon

CSV_PATH = '../results/benchmark_results.csv'
FIGURES_DIR = '../figures'
import os; os.makedirs(FIGURES_DIR, exist_ok=True)

df   = pd.read_csv(CSV_PATH)
cold = df[df['RunType'] == 'COLD'].copy()

# Hierarchy model → (query_col, display_label, strategy_label)
HIER_MAP = {
    'B1': ('Q3_B1',       'B1 — Flat (control)',                    'Flat'),
    'B2': ('Q3_B2',       'B2 — Shallow: /year=',                   '/year='),
    'B3': ('Q3_B3',       'B3 — Deep: /year=/month=',               '/year=/month='),
    'B4_month': ('Q3_B4_month', 'B4 — 3-level: same-month query',   '/year=/month=/day=\n(month query)'),
    'B4_day':   ('Q3_B4_day',  'B4 — 3-level: single-day query',    '/year=/month=/day=\n(day query)'),
}

C1_MAP = {
    'Q3_Year':  'C1 — nested key\n(year filter)',
    'Q3_Month': 'C1 — top-level key\n(month filter)',
    'Q3_Both':  'C1 — both keys',
    'Q3_B3':    'B3 — direct path\n(year + month)',
}

print('Ready.')

## 1  Table 2 — Hierarchy depth results, SF-10

In [ ]:
rows = []
b1_median = cold[(cold['SF']==10)&(cold['Model']=='B1')&(cold['Query']=='Q3_B1')]['WallClock_ms'].median()

for key, (query, label, strategy) in HIER_MAP.items():
    model = 'B4' if key.startswith('B4') else key
    sub = cold[(cold['SF']==10) & (cold['Model']==model) & (cold['Query']==query)]['WallClock_ms']
    med = sub.median()
    std = sub.std()
    speedup = b1_median / med
    rows.append({'Model/Query': label, 'Strategy': strategy,
                 'Median (ms)': round(med, 1), 'Std (ms)': round(std, 1),
                 'Speedup vs B1': round(speedup, 2)})

table2 = pd.DataFrame(rows)
print('Table 2 — Partition depth impact (SF-10):')
print(table2.to_string(index=False))
table2.to_csv(f'{FIGURES_DIR}/table2_rq2_hierarchy_sf10.csv', index=False)

## 2  Table 3 — Key ordering results, SF-10

In [ ]:
rows3 = []
for query, label in C1_MAP.items():
    model = 'B3' if query == 'Q3_B3' else 'C1'
    sub = cold[(cold['SF']==10) & (cold['Model']==model) & (cold['Query']==query)]['WallClock_ms']
    rows3.append({'Model': model, 'Filter / Operation': label,
                  'Median (ms)': round(sub.median(), 1),
                  'Std (ms)': round(sub.std(), 1)})

table3 = pd.DataFrame(rows3)
print('Table 3 — Key ordering impact (SF-10):')
print(table3.to_string(index=False))
table3.to_csv(f'{FIGURES_DIR}/table3_rq3_keyorder_sf10.csv', index=False)

# Compute penalty %
c1_month = cold[(cold['SF']==10)&(cold['Model']=='C1')&(cold['Query']=='Q3_Month')]['WallClock_ms'].median()
c1_year  = cold[(cold['SF']==10)&(cold['Model']=='C1')&(cold['Query']=='Q3_Year')]['WallClock_ms'].median()
c1_both  = cold[(cold['SF']==10)&(cold['Model']=='C1')&(cold['Query']=='Q3_Both')]['WallClock_ms'].median()
b3_both  = cold[(cold['SF']==10)&(cold['Model']=='B3')&(cold['Query']=='Q3_B3')]['WallClock_ms'].median()

print(f'\nNested-key penalty (year vs month filter): {(c1_year/c1_month - 1)*100:.1f}%')
print(f'C1(both) vs B3(both):                     {(c1_both/b3_both - 1)*100:.1f}% overhead')

## 3  Figure — Hierarchy depth: speedup/slowdown bars (SF-10 and SF-1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

hier_order = ['B1', 'B2', 'B3', 'B4_month', 'B4_day']
hier_xlabels = ['B1\nFlat', 'B2\n/year=', 'B3\n/year=\n/month=',
                'B4\n3-level\n(month Q)', 'B4\n3-level\n(day Q)']
colors = ['#808080', '#2E75B6', '#70AD47', '#C00000', '#C00000']

for ax, sf in zip(axes, [10, 1]):
    b1_q = 'Q3_B1'
    b1_med_sf = cold[(cold['SF']==sf)&(cold['Model']=='B1')&(cold['Query']==b1_q)]['WallClock_ms'].median()

    medians, stds, speedups = [], [], []
    for key in hier_order:
        model = 'B4' if key.startswith('B4') else key
        query = HIER_MAP[key][0]
        sub = cold[(cold['SF']==sf) & (cold['Model']==model) & (cold['Query']==query)]['WallClock_ms']
        medians.append(sub.median())
        stds.append(sub.std())
        speedups.append(b1_med_sf / sub.median())

    x = range(len(hier_order))
    bars = ax.bar(x, medians, yerr=stds, capsize=4, color=colors, alpha=0.85)
    ax.set_yscale('log')
    ax.axhline(y=b1_med_sf, color='grey', linestyle='--', linewidth=1.0, alpha=0.6)
    ax.set_title(f'Hierarchy depth — SF-{sf}', fontweight='bold')
    ax.set_xticks(list(x))
    ax.set_xticklabels(hier_xlabels, fontsize=8)
    ax.set_ylabel('Median wall-clock (ms)')
    ax.grid(axis='y', linestyle='--', alpha=0.3)

    for bar, sp in zip(bars, speedups):
        label = f'{sp:.2f}×' if sp >= 0.1 else f'1/{1/sp:.0f}×'
        ypos = bar.get_height() * 1.15
        ax.text(bar.get_x() + bar.get_width()/2, ypos, label,
                ha='center', va='bottom', fontsize=8, fontweight='bold')

fig.suptitle('Partition hierarchy depth effect on selective query latency\n'
             'Values show speedup (>1) or slowdown (<1) vs B1 flat control',
             fontsize=11)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig_rq2_hierarchy_depth.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES_DIR}/fig_rq2_hierarchy_depth.png', dpi=200, bbox_inches='tight')
plt.show()

## 4  Figure — Key ordering: C1 variants vs B3 reference (SF-10 and SF-1)

In [ ]:
order_entries = [
    ('C1', 'Q3_Year',  'C1\nnested key\n(year)', '#E2EFDA', '#70AD47'),
    ('C1', 'Q3_Month', 'C1\ntop-level key\n(month)', '#DEEBF7', '#2E75B6'),
    ('C1', 'Q3_Both',  'C1\nboth keys', '#FFF2CC', '#FFC000'),
    ('B3', 'Q3_B3',    'B3\ndirect path\n/year=/month=', '#F4CCCC', '#C00000'),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, sf in zip(axes, [10, 1]):
    labels, vals, errs, face_colors, edge_colors = [], [], [], [], []
    for model, query, label, fc, ec in order_entries:
        sub = cold[(cold['SF']==sf)&(cold['Model']==model)&(cold['Query']==query)]['WallClock_ms']
        labels.append(label)
        vals.append(sub.median())
        errs.append(sub.std())
        face_colors.append(fc)
        edge_colors.append(ec)

    x = range(len(labels))
    bars = ax.bar(x, vals, yerr=errs, capsize=4,
                  color=face_colors, edgecolor=edge_colors, linewidth=1.5, alpha=0.9)
    ax.set_title(f'Key ordering effect — SF-{sf}', fontweight='bold')
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel('Median wall-clock (ms)')
    ax.grid(axis='y', linestyle='--', alpha=0.3)

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(errs)*0.05,
                f'{val:.0f}', ha='center', va='bottom', fontsize=9)

fig.suptitle('Partition key ordering: C1 (/month=/year=) vs B3 (/year=/month=)',
             fontsize=11)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig_rq3_key_ordering.pdf', bbox_inches='tight')
plt.savefig(f'{FIGURES_DIR}/fig_rq3_key_ordering.png', dpi=200, bbox_inches='tight')
plt.show()

## 5  Wilcoxon tests — RQ2 hierarchy comparisons (SF-10)

In [ ]:
def wtest(name_a, model_a, query_a, name_b, model_b, query_b, sf=10):
    a = cold[(cold['SF']==sf)&(cold['Model']==model_a)&(cold['Query']==query_a)]['WallClock_ms'].values
    b = cold[(cold['SF']==sf)&(cold['Model']==model_b)&(cold['Query']==query_b)]['WallClock_ms'].values
    stat, p = wilcoxon(a, b)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f'  {name_a:<30} vs {name_b:<30}  '
          f'med={np.median(a):.1f} vs {np.median(b):.1f} ms   '
          f'p={p:.6f}  {sig}')

print('=== RQ2: Hierarchy depth (SF-10) ===')
wtest('B1 (flat)',     'B1','Q3_B1', 'B2 (/year=)',     'B2','Q3_B2')
wtest('B1 (flat)',     'B1','Q3_B1', 'B3 (/year=/month=)','B3','Q3_B3')
wtest('B2 (/year=)',   'B2','Q3_B2', 'B3 (/year=/month=)','B3','Q3_B3')
wtest('B3 (/year=/mo)','B3','Q3_B3', 'B4 (3-level month)','B4','Q3_B4_month')
wtest('B1 (flat)',     'B1','Q3_B1', 'B4 (3-level month)','B4','Q3_B4_month')
wtest('B1 (flat)',     'B1','Q3_B1', 'B4 (3-level day)',  'B4','Q3_B4_day')

print()
print('=== RQ2: Hierarchy depth (SF-1) ===')
wtest('B1 SF-1', 'B1','Q3_B1', 'B3 SF-1', 'B3','Q3_B3', sf=1)
wtest('B1 SF-1', 'B1','Q3_B1', 'B4 month SF-1', 'B4','Q3_B4_month', sf=1)

print()
print('=== RQ3: Key ordering (SF-10) ===')
wtest('C1 top-level (month)', 'C1','Q3_Month', 'C1 nested (year)', 'C1','Q3_Year')
wtest('C1 both keys',         'C1','Q3_Both',  'B3 direct path',   'B3','Q3_B3')
wtest('C1 both keys',         'C1','Q3_Both',  'C1 top-level',     'C1','Q3_Month')

## 6  SF-1 vs SF-10 hierarchy comparison summary

In [ ]:
print('Hierarchy model medians — SF-1 vs SF-10:')
print(f'{"Model/Query":<28} {"SF-1 (ms)":>12} {"SF-10 (ms)":>12} {"Ratio":>8}')
print('-' * 65)
for key, (query, label, _) in HIER_MAP.items():
    model = 'B4' if key.startswith('B4') else key
    sf1  = cold[(cold['SF']==1) &(cold['Model']==model)&(cold['Query']==query)]['WallClock_ms'].median()
    sf10 = cold[(cold['SF']==10)&(cold['Model']==model)&(cold['Query']==query)]['WallClock_ms'].median()
    ratio = sf10 / sf1
    print(f'{label:<28} {sf1:>12.1f} {sf10:>12.1f} {ratio:>7.1f}×')